# 02. Hybrid embedding과 text-only loss

목표: video embedding을 언어 모델 차원으로 투영하고 optional OCR·intent token과 연결한 뒤, loss를 intent text에만 적용하는 UI-JEPA의 두 번째 단계를 작은 숫자로 재현합니다.

In [ ]:
video_tokens = [
    [0.2, 0.1, 0.4],
    [0.7, 0.3, 0.1],
    [0.1, 0.8, 0.2],
]
projection = [
    [0.5, 0.0, 0.2, 0.1],
    [0.1, 0.6, 0.0, 0.2],
    [0.0, 0.2, 0.7, 0.1],
]

def project(vector, matrix):
    return [sum(vector[i] * matrix[i][j] for i in range(len(vector)))
            for j in range(len(matrix[0]))]

projected_video = [project(token, projection) for token in video_tokens]
print('projected video shape:', (len(projected_video), len(projected_video[0])))
print('first projected token:', projected_video[0])
assert len(projected_video[0]) == 4

In [ ]:
SEP, EOS, IGNORE = 9001, 9002, -100
ocr_tokens = [301, 302, 303]       # 예: 마지막 frame에서 추출한 text
intent_tokens = [501, 502, 503, EOS]
video_placeholders = ['VIDEO'] * len(projected_video)
sequence = video_placeholders + [SEP] + ocr_tokens + [SEP] + intent_tokens

# video·separator·OCR는 condition일 뿐 정답 생성 loss의 대상이 아닙니다.
prefix_length = len(video_placeholders) + 1 + len(ocr_tokens) + 1
labels = [IGNORE] * prefix_length + intent_tokens
print('sequence:', sequence)
print('labels:  ', labels)
assert len(sequence) == len(labels)
assert all(label == IGNORE for label in labels[:prefix_length])
assert labels[prefix_length:] == intent_tokens

In [ ]:
from math import log

# 실제 softmax 대신 정답 token에 할당한 확률로 masked cross-entropy를 설명합니다.
correct_token_probabilities = [0.70, 0.55, 0.80, 0.90]
text_loss = -sum(log(p) for p in correct_token_probabilities) / len(correct_token_probabilities)
print('intent text-only loss:', round(text_loss, 4))
assert text_loss > 0

In [ ]:
def lora_parameter_count(input_dim, output_dim, rank):
    # 기존 weight(input_dim * output_dim)는 고정하고 A와 B만 학습합니다.
    return rank * (input_dim + output_dim)

full = 3072 * 3072
lora = lora_parameter_count(3072, 3072, rank=16)
print({'full weight': full, 'LoRA trainable': lora, 'ratio': f'{lora / full:.2%}'})
assert lora < full

## 실무 주의점

OCR에는 message, 연락처, 계정 정보가 포함될 수 있습니다. tokenization 전에 redaction하고, video prefix의 padding mask와 text label mask를 혼동하지 않도록 unit test를 작성해야 합니다.